In [1]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

ModuleNotFoundError: No module named 'pandas'

In [ ]:
print(os.listdir(r"C:\Users\Arjun Kaushik\OneDrive\Desktop\Intrusion Detection System"))

In [ ]:
with open(r"C:\Users\Arjun Kaushik\OneDrive\Desktop\Intrusion Detection System\kddcup.names", 'r') as f:
    data = f.read()
    print(data)

In [ ]:
cols="""duration,
protocol_type,
service,
flag,
src_bytes,
dst_bytes,
land,
wrong_fragment,
urgent,
hot,
num_failed_logins,
logged_in,
num_compromised,
root_shell,
su_attempted,
num_root,
num_file_creations,
num_shells,
num_access_files,
num_outbound_cmds,
is_host_login,
is_guest_login,
count,
srv_count,
serror_rate,
srv_serror_rate,
rerror_rate,
srv_rerror_rate,
same_srv_rate,
diff_srv_rate,
srv_diff_host_rate,
dst_host_count,
dst_host_srv_count,
dst_host_same_srv_rate,
dst_host_diff_srv_rate,
dst_host_same_src_port_rate,
dst_host_srv_diff_host_rate,
dst_host_serror_rate,
dst_host_srv_serror_rate,
dst_host_rerror_rate,
dst_host_srv_rerror_rate"""

columns=[]
for c in cols.split(','):
    if(c.strip()):
       columns.append(c.strip())

columns.append('target')
#print(columns)
print(len(columns))

In [ ]:
with open(r"C:\Users\Arjun Kaushik\OneDrive\Desktop\Intrusion Detection System\training_attack_types.txt",'r') as f:
    print(f.read())

In [ ]:
attacks_types = {
    'normal': 'normal',
'back': 'dos',
'buffer_overflow': 'u2r',
'ftp_write': 'r2l',
'guess_passwd': 'r2l',
'imap': 'r2l',
'ipsweep': 'probe',
'land': 'dos',
'loadmodule': 'u2r',
'multihop': 'r2l',
'neptune': 'dos',
'nmap': 'probe',
'perl': 'u2r',
'phf': 'r2l',
'pod': 'dos',
'portsweep': 'probe',
'rootkit': 'u2r',
'satan': 'probe',
'smurf': 'dos',
'spy': 'r2l',
'teardrop': 'dos',
'warezclient': 'r2l',
'warezmaster': 'r2l',
}


READING DATASET

In [ ]:
path = r"C:\Users\Arjun Kaushik\OneDrive\Desktop\Intrusion Detection System\kddcup.data_10_percent.gz"
df = pd.read_csv(path,names=columns)

#Adding Attack Type column
df['Attack Type'] = df.target.apply(lambda r:attacks_types[r[:-1]])

df.head()

In [ ]:
df.shape

In [ ]:
df['target'].value_counts()

In [ ]:
df['Attack Type'].value_counts()

In [ ]:
df.dtypes

DATA PREPROCESSING

In [ ]:
df.isnull().sum()

In [ ]:
#Finding categorical features
num_cols = df._get_numeric_data().columns

cate_cols = list(set(df.columns)-set(num_cols))
cate_cols.remove('target')
cate_cols.remove('Attack Type')

cate_cols

CATEGORICAL FEATURES DISTRIBUTION

In [ ]:
#Visualization
def bar_graph(feature):
    df[feature].value_counts().plot(kind="bar")

In [ ]:
bar_graph('protocol_type')

Protocol type: We notice that ICMP is the most present in the used data, then TCP and almost 20000 packets of UDP type

In [ ]:
plt.figure(figsize=(15,3))
bar_graph('service')

In [ ]:
bar_graph('flag')

In [ ]:
bar_graph('logged_in')

logged_in (1 if successfully logged in; 0 otherwise): We notice that just 70000 packets are successfully logged in.

TARGET FEATURE DISTRIBUTION

In [ ]:
bar_graph('target')

Attack Type(The attack types grouped by attack, it's what we will predict)

In [ ]:
bar_graph('Attack Type')

In [ ]:
df.columns

DATA CORRELATION

In [ ]:
df['num_root'].corr(df['num_compromised'])

In [ ]:
df['srv_serror_rate'].corr(df['serror_rate'])

In [ ]:
df['srv_count'].corr(df['count'])

In [ ]:
df['srv_rerror_rate'].corr(df['rerror_rate'])

In [ ]:
df['dst_host_same_srv_rate'].corr(df['dst_host_srv_count'])

In [ ]:
df['dst_host_srv_serror_rate'].corr(df['dst_host_serror_rate'])

In [ ]:
df['dst_host_srv_rerror_rate'].corr(df['dst_host_rerror_rate'])

In [ ]:
df['dst_host_same_srv_rate'].corr(df['same_srv_rate'])

In [ ]:
df['dst_host_srv_count'].corr(df['same_srv_rate'])

In [ ]:
df['dst_host_same_src_port_rate'].corr(df['srv_count'])

In [ ]:
df['dst_host_serror_rate'].corr(df['serror_rate'])

In [ ]:
df['dst_host_serror_rate'].corr(df['srv_serror_rate'])

In [ ]:
df['dst_host_srv_serror_rate'].corr(df['serror_rate'])

In [ ]:
df['dst_host_srv_serror_rate'].corr(df['srv_serror_rate'])

In [ ]:
df['dst_host_rerror_rate'].corr(df['rerror_rate'])

In [ ]:
df['dst_host_rerror_rate'].corr(df['srv_rerror_rate'])

In [ ]:
df['dst_host_srv_rerror_rate'].corr(df['rerror_rate'])

In [ ]:
df['dst_host_srv_rerror_rate'].corr(df['srv_rerror_rate'])

In [ ]:
#This variable is highly correlated with num_compromised and should be ignored for analysis.
#(Correlation = 0.9938277978738366)
df.drop('num_root',axis = 1,inplace = True)

#This variable is highly correlated with serror_rate and should be ignored for analysis.
#(Correlation = 0.9983615072725952)
df.drop('srv_serror_rate',axis = 1,inplace = True)

#This variable is highly correlated with rerror_rate and should be ignored for analysis.
#(Correlation = 0.9947309539817937)
df.drop('srv_rerror_rate',axis = 1, inplace=True)

#This variable is highly correlated with srv_serror_rate and should be ignored for analysis.
#(Correlation = 0.9993041091850098)
df.drop('dst_host_srv_serror_rate',axis = 1, inplace=True)

#This variable is highly correlated with rerror_rate and should be ignored for analysis.
#(Correlation = 0.9869947924956001)
df.drop('dst_host_serror_rate',axis = 1, inplace=True)

#This variable is highly correlated with srv_rerror_rate and should be ignored for analysis.
#(Correlation = 0.9821663427308375)
df.drop('dst_host_rerror_rate',axis = 1, inplace=True)

#This variable is highly correlated with rerror_rate and should be ignored for analysis.
#(Correlation = 0.9851995540751249)
df.drop('dst_host_srv_rerror_rate',axis = 1, inplace=True)

#This variable is highly correlated with srv_rerror_rate and should be ignored for analysis.
#(Correlation = 0.9865705438845669)
df.drop('dst_host_same_srv_rate',axis = 1, inplace=True)

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.columns

In [ ]:
df_std = df.std()
df_std = df_std.sort_values(ascending = True)
df_std

FEATURE MAPPING

In [ ]:
df['protocol_type'].value_counts()

In [ ]:
#protocol_type feature mapping
pmap = {'icmp':0,'tcp':1,'udp':2}
df['protocol_type'] = df['protocol_type'].map(pmap)

In [ ]:
df['flag'].value_counts()

In [ ]:
#flag feature mapping
fmap = {'SF':0,'S0':1,'REJ':2,'RSTR':3,'RSTO':4,'SH':5 ,'S1':6 ,'S2':7,'RSTOS0':8,'S3':9 ,'OTH':10}
df['flag'] = df['flag'].map(fmap)

In [ ]:
df.head()

In [ ]:
df.drop('service',axis = 1,inplace= True)

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.dtypes

MODELLING

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler

In [ ]:
df = df.drop(['target',], axis=1)
print(df.shape)

# Target variable and train set
y = df[['Attack Type']]
X = df.drop(['Attack Type',], axis=1)

sc = MinMaxScaler()
X = sc.fit_transform(X)

# Split test and train data 
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
print(X_train.shape, X_test.shape)
print(y_train.shape, y_test.shape)

GAUSSIAN NAIVE BAYES

In [ ]:
# Gaussian Naive Bayes
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import accuracy_score

In [ ]:
clfg = GaussianNB()

In [ ]:
start_time = time.time()
clfg.fit(X_train, y_train.values.ravel())
end_time = time.time()

In [ ]:
print("Training time: ",end_time-start_time)

In [ ]:
start_time = time.time()
y_test_pred = clfg.predict(X_train)
end_time = time.time()

In [ ]:
print("Testing time: ",end_time-start_time)

In [ ]:
print("Train score is:", clfg.score(X_train, y_train))
print("Test score is:",clfg.score(X_test,y_test))

DECISION TREE

In [ ]:
#Decision Tree 
from sklearn.tree import DecisionTreeClassifier

In [ ]:
clfd = DecisionTreeClassifier(criterion="entropy", max_depth = 4)

In [ ]:
start_time = time.time()
clfd.fit(X_train, y_train.values.ravel())
end_time = time.time()

In [ ]:
print("Training time: ",end_time-start_time)

In [ ]:
start_time = time.time()
y_test_pred = clfd.predict(X_train)
end_time = time.time()

In [ ]:
print("Testing time: ",end_time-start_time)

In [ ]:
print("Train score is:", clfd.score(X_train, y_train))
print("Test score is:",clfd.score(X_test,y_test))

RANDOM FOREST

In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
clfr = RandomForestClassifier(n_estimators=30)

In [ ]:
start_time = time.time()
clfr.fit(X_train, y_train.values.ravel())
end_time = time.time()

In [ ]:
print("Training time: ",end_time-start_time)

In [ ]:
start_time = time.time()
y_test_pred = clfd.predict(X_train)
end_time = time.time()

In [ ]:
print("Testing time: ",end_time-start_time)

In [ ]:
print("Train score is:", clfr.score(X_train, y_train))
print("Test score is:",clfr.score(X_test,y_test))

SUPPORT VECTOR MACHINE

In [ ]:
from sklearn.svm import SVC

In [ ]:
clfs = SVC(gamma = 'scale')

In [ ]:
start_time = time.time()
clfs.fit(X_train, y_train.values.ravel())
end_time = time.time()

In [ ]:
print("Training time: ",end_time-start_time)

In [ ]:
start_time = time.time()
y_test_pred = clfs.predict(X_train)
end_time = time.time()

In [ ]:
print("Testing time: ",end_time-start_time)

In [ ]:
print("Train score is:", clfs.score(X_train, y_train))
print("Test score is:",clfs.score(X_test,y_test))

LOGISTIC REGRESSION

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
clfl = LogisticRegression(max_iter=1200000)

In [ ]:
start_time = time.time()
clfl.fit(X_train, y_train.values.ravel())
end_time = time.time()

In [ ]:
print("Training time: ",end_time-start_time)

In [ ]:
start_time = time.time()
y_test_pred = clfl.predict(X_train)
end_time = time.time()

In [ ]:
print("Testing time: ",end_time-start_time)

In [ ]:
print("Train score is:", clfl.score(X_train, y_train))
print("Test score is:",clfl.score(X_test,y_test))

TRAINING ACCURACY

In [ ]:
names = ['NB','DT','RF','SVM','LR']
values = [87.951,99.058,99.997,99.875,99.352]
f = plt.figure(figsize=(15,3),num=10)
plt.subplot(131)
plt.bar(names,values)

In [ ]:
f.savefig('training_accuracy_figure.png',bbox_inches='tight')

TESTING ACCURACY

In [ ]:
names = ['NB','DT','RF','SVM','LR']
values = [87.903,99.052,99.966,99.879,99.352]
f = plt.figure(figsize=(15,3),num=10)
plt.subplot(131)
plt.bar(names,values)

In [ ]:
f.savefig('test_accuracy_figure.png',bbox_inches='tight')

TRAINING TIME

In [ ]:
names = ['NB','DT','RF','SVM','LR']
values = [1.28314,2.27796,16.35494,216.53651,97.92668]
f = plt.figure(figsize=(15,3),num=10)
plt.subplot(131)
plt.bar(names,values)

In [ ]:
f.savefig('train_time_figure.png',bbox_inches='tight')

TESTING TIME

In [ ]:
names = ['NB','DT','RF','SVM','LR']
values = [1.60423,0.14665,0.12128,131.44433,0.08549]
f = plt.figure(figsize=(15,3),num=10)
plt.subplot(131)
plt.bar(names,values)

In [ ]:
f.savefig('test_time_figure.png',bbox_inches='tight')

In [ ]:
pip install joblib

In [ ]:
import joblib

joblib.dump(clfr, "model.pkl")

print("Model saved successfully!")

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import joblib

# Train model
clfr = RandomForestClassifier(n_estimators=30)
clfr.fit(X_train, y_train.values.ravel())

# Save model
joblib.dump(clfr, "model.pkl")

print("Model saved successfully!")

In [ ]:
!pip install scikit-learn


In [ ]:
from sklearn.ensemble import RandomForestClassifier

In [ ]:
import sys
!{sys.executable} -m pip install scikit-learn

In [ ]:
!pip3 install scikit-learn

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import joblib

# Train model
clfr = RandomForestClassifier(n_estimators=30)
clfr.fit(X_train, y_train.values.ravel())

# Save model
joblib.dump(clfr, "model.pkl")

print("Model saved successfully!")

In [ ]:
!pip install scikit-learn joblib

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import joblib

print("Working ✅")

In [2]:
from sklearn.ensemble import RandomForestClassifier
import joblib

print("Working ✅")

Working ✅


In [3]:
from sklearn.ensemble import RandomForestClassifier
import joblib

# Train model
clfr = RandomForestClassifier(n_estimators=30)
clfr.fit(X_train, y_train.values.ravel())

# Save model
joblib.dump(clfr, "model.pkl")

print("Model saved successfully!")

NameError: name 'X_train' is not defined

In [6]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import joblib

# Load dataset
data = pd.read_csv("kddcup.data_10_percent.gz", header=None)

# Select only required features (same as frontend)
X = data.iloc[:, [0, 4, 5]]   # duration, src_bytes, dst_bytes
y = data.iloc[:, -1]

# Convert labels (normal vs attack)
y = y.apply(lambda x: 0 if x == 'normal.' else 1)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Train model
clfr = RandomForestClassifier(n_estimators=30)
clfr.fit(X_train, y_train)

# Save model
joblib.dump(clfr, "model.pkl")

print("Model ready ✅")

ModuleNotFoundError: No module named 'pandas'

In [5]:
!pip install pandas


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import joblib

# Load dataset
data = pd.read_csv("kddcup.data_10_percent.gz", header=None)

# Select only required features (same as frontend)
X = data.iloc[:, [0, 4, 5]]   # duration, src_bytes, dst_bytes
y = data.iloc[:, -1]

# Convert labels (normal vs attack)
y = y.apply(lambda x: 0 if x == 'normal.' else 1)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Train model
clfr = RandomForestClassifier(n_estimators=30)
clfr.fit(X_train, y_train)

# Save model
joblib.dump(clfr, "model.pkl")

print("Model ready ✅")

ModuleNotFoundError: No module named 'pandas'

In [8]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
import joblib

# Load dataset
data = pd.read_csv("kddcup.data_10_percent.gz", header=None)

# Select only required features (same as frontend)
X = data.iloc[:, [0, 4, 5]]   # duration, src_bytes, dst_bytes
y = data.iloc[:, -1]

# Convert labels (normal vs attack)
y = y.apply(lambda x: 0 if x == 'normal.' else 1)

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# Train model
clfr = RandomForestClassifier(n_estimators=30)
clfr.fit(X_train, y_train)

# Save model
joblib.dump(clfr, "model.pkl")

print("Model ready ✅")

ModuleNotFoundError: No module named 'pandas'